# Chapter 1 — Introduction to Econometrics
## University of Neuchâtel — Python for Econometrics

**What is econometrics?** The application of statistical methods to economic data
to test theories and estimate relationships between economic variables.

Topics:
1. Variable types and data exploration
2. Descriptive statistics: mean, median, standard deviation
3. Covariance and correlation — manual vs numpy
4. Simple OLS: slope, intercept, fitted values, residuals
5. SST, SSE, SSR and R-squared
6. Comparing manual results with statsmodels

**Format:** One cell per step. Manual calculation first, then built-in/statsmodels, then `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Load Dataset: `wage1`

In [ ]:
df = woo.dataWoo('wage1')
print('Shape:', df.shape)
print(df[['wage','educ','exper','female','tenure']].head(10))

In [ ]:
print('Descriptive statistics:')
print(df[['wage','educ','exper','female','tenure']].describe().round(3))

## 2 — Variable Types

**Continuous:** can take many values — `wage`, `educ`, `exper`

**Binary / Dummy:** only 0 or 1 — `female`, `married`, `smsa`

Understanding variable types is essential before choosing a regression model.

In [ ]:
print('Continuous: wage')
print(df['wage'].describe())
print('\nBinary: female')
print(df['female'].value_counts())
print('Share female:', df['female'].mean().round(3))

## 3 — Descriptive Statistics: Manual vs Pandas

### Step 3.1 — Mean manually

In [ ]:
n = len(df)
wage_mean_manual = df['wage'].sum() / n
print(f'Manual mean wage: {wage_mean_manual:.4f}')

In [ ]:
wage_mean_pd = df['wage'].mean()
print(f'Pandas mean wage: {wage_mean_pd:.4f}')
print(f'Match: {np.isclose(wage_mean_manual, wage_mean_pd)}')

### Step 3.2 — Variance and standard deviation manually

In [ ]:
wage_var_manual = np.sum((df['wage'] - wage_mean_manual)**2) / (n - 1)
wage_std_manual = np.sqrt(wage_var_manual)
print(f'Manual std wage: {wage_std_manual:.4f}')

In [ ]:
wage_std_pd = df['wage'].std(ddof=1)
print(f'Pandas std wage: {wage_std_pd:.4f}')
print(f'Match: {np.isclose(wage_std_manual, wage_std_pd)}')

## 4 — Distribution of Wage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].hist(df['wage'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(wage_mean_manual, color='red', linestyle='--', label=f'Mean={wage_mean_manual:.2f}')
axes[0].set_xlabel('Wage ($/hr)')
axes[0].set_title('Distribution of Wages')
axes[0].legend()
axes[1].hist(np.log(df['wage']), bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('log(Wage)')
axes[1].set_title('Distribution of log(Wage) — more symmetric')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch01_01_wage_dist.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5 — Scatter Plot: Education vs Wage

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df['educ'], df['wage'], alpha=0.4, s=20)
ax.set_xlabel('Years of Education')
ax.set_ylabel('Wage ($/hr)')
ax.set_title('Education vs Wage')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch01_02_scatter_educ_wage.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Visual: positive association between education and wages')

## 6 — Covariance and Correlation

### Step 6.1 — Covariance manually

In [ ]:
x = df['educ'].values
y = df['wage'].values
x_mean = x.mean()
y_mean = y.mean()
cov_manual = np.sum((x - x_mean) * (y - y_mean)) / (n - 1)
print(f'Manual covariance(educ, wage): {cov_manual:.4f}')

In [ ]:
cov_np = np.cov(x, y)[0, 1]
print(f'numpy cov:                     {cov_np:.4f}')
print(f'Match: {np.isclose(cov_manual, cov_np)}')

### Step 6.2 — Correlation manually

In [ ]:
x_std = x.std(ddof=1)
y_std = y.std(ddof=1)
corr_manual = cov_manual / (x_std * y_std)
print(f'Manual correlation(educ, wage): {corr_manual:.4f}')

In [ ]:
corr_np = np.corrcoef(x, y)[0, 1]
print(f'numpy corrcoef:                 {corr_np:.4f}')
print(f'Match: {np.isclose(corr_manual, corr_np)}')
print(f'Interpretation: r={corr_manual:.3f} → positive, moderate linear relationship')

## 7 — Simple OLS Regression

**Model:** $wage_i = \beta_0 + \beta_1 \cdot educ_i + u_i$

**OLS formulas:**

$$\hat{\beta}_1 = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^n (x_i - \bar{x})^2}$$

$$\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$

### Step 7.1 — Compute OLS slope manually

In [ ]:
x_dev = x - x_mean
y_dev = y - y_mean
b1_manual = np.sum(x_dev * y_dev) / np.sum(x_dev**2)
print(f'OLS slope b1 (manual): {b1_manual:.4f}')
print(f'Interpretation: one extra year of education raises wage by ${b1_manual:.4f}/hr')

### Step 7.2 — Compute OLS intercept manually

In [ ]:
b0_manual = y_mean - b1_manual * x_mean
print(f'OLS intercept b0 (manual): {b0_manual:.4f}')
print(f'Predicted wage with 0 years of education: ${b0_manual:.2f}/hr (extrapolation only)')

### Step 7.3 — Compute fitted values manually

In [ ]:
y_hat = b0_manual + b1_manual * x
print('First 5 fitted values:')
for i in range(5):
    print(f'  educ={x[i]}, wage={y[i]:.2f}, fitted={y_hat[i]:.2f}')

### Step 7.4 — Compute residuals manually

In [ ]:
e = y - y_hat
print('First 5 residuals:')
for i in range(5):
    print(f'  e[{i}] = {y[i]:.2f} - {y_hat[i]:.2f} = {e[i]:.2f}')
print(f'\nSum of residuals (should be ~0): {e.sum():.10f}')
print(f'Sum of x*e (should be ~0):       {np.sum(x*e):.10f}')

## 8 — Goodness of Fit: SST, SSE, SSR, R²

**Total Sum of Squares:** $SST = \sum(y_i - \bar{y})^2$

**Explained Sum of Squares:** $SSE = \sum(\hat{y}_i - \bar{y})^2$

**Residual Sum of Squares:** $SSR = \sum \hat{u}_i^2$

**Identity:** $SST = SSE + SSR$

$$R^2 = \frac{SSE}{SST} = 1 - \frac{SSR}{SST}$$

### Step 8.1 — Compute SST, SSE, SSR manually

In [ ]:
SST = np.sum((y - y_mean)**2)
SSR = np.sum(e**2)
SSE = np.sum((y_hat - y_mean)**2)
print(f'SST = {SST:.4f}')
print(f'SSE = {SSE:.4f}')
print(f'SSR = {SSR:.4f}')
print(f'SSE + SSR = {SSE + SSR:.4f}  (should equal SST)')
print(f'Match: {np.isclose(SST, SSE + SSR)}')

### Step 8.2 — Compute R² manually

In [ ]:
R2_manual = 1 - SSR / SST
print(f'R² (manual) = {R2_manual:.4f}')
print(f'Interpretation: {100*R2_manual:.1f}% of wage variation explained by education')

## 9 — Statsmodels OLS

In [ ]:
res = smf.ols('wage ~ educ', data=df).fit()
print(res.summary())

### Step 9.1 — Compare ALL quantities with np.isclose

In [ ]:
print('=== Manual vs Statsmodels Comparison ===')
print(f'b0:  manual={b0_manual:.4f}, sm={res.params["Intercept"]:.4f}, match={np.isclose(b0_manual, res.params["Intercept"])}')
print(f'b1:  manual={b1_manual:.4f}, sm={res.params["educ"]:.4f},      match={np.isclose(b1_manual, res.params["educ"])}')
print(f'R2:  manual={R2_manual:.4f},  sm={res.rsquared:.4f},             match={np.isclose(R2_manual, res.rsquared)}')
print(f'SSR: manual={SSR:.4f},      sm={res.ssr:.4f},             match={np.isclose(SSR, res.ssr)}')

## 10 — Regression Line Plot

In [ ]:
educ_range = np.linspace(df['educ'].min(), df['educ'].max(), 100)
wage_pred  = b0_manual + b1_manual * educ_range
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df['educ'], df['wage'], alpha=0.3, s=20, label='Data')
ax.plot(educ_range, wage_pred, 'r-', linewidth=2, label=f'OLS: wage = {b0_manual:.2f} + {b1_manual:.2f}*educ')
ax.set_xlabel('Years of Education')
ax.set_ylabel('Wage ($/hr)')
ax.set_title('OLS Regression Line: wage ~ educ')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch01_03_regression_line.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11 — Residual Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.scatter(y_hat, e, alpha=0.3, s=20)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Fitted Values')
ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted Values')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch01_04_residuals.png'), dpi=150, bbox_inches='tight')
plt.show()
print('A good model: residuals randomly scattered around 0 with no pattern')

## 12 — Summary Table

In [ ]:
sumtab = pd.DataFrame({
    'Quantity': ['b0 (intercept)','b1 (educ slope)','R-squared','SSR'],
    'Manual':   [b0_manual, b1_manual, R2_manual, SSR],
    'Statsmodels': [res.params['Intercept'], res.params['educ'], res.rsquared, res.ssr],
    'Match': [
        np.isclose(b0_manual, res.params['Intercept']),
        np.isclose(b1_manual, res.params['educ']),
        np.isclose(R2_manual, res.rsquared),
        np.isclose(SSR, res.ssr)
    ]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch01_ols_comparison.csv'), index=False)
print('Saved table_ch01_ols_comparison.csv')